<a href="https://colab.research.google.com/github/Ramdharshan2007/DAA-Lab-Experiment/blob/main/8A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
import copy

class Node:
    """
    Represents a state (node) in the State Space Tree for the TSP.
    """
    def __init__(self, matrix, path, bound, level, current_city):
        self.matrix = matrix        # The reduced cost matrix for this state
        self.path = path            # Cities visited so far
        self.bound = bound          # The lower bound cost of this node
        self.level = level          # Number of edges in the path so far
        self.current_city = current_city # The city we are currently positioned at

    # Define the less-than operator to allow heapq to sort Nodes by their bound
    def __lt__(self, other):
        return self.bound < other.bound

def reduce_matrix(matrix):
    """
    Reduces the given matrix by subtracting the minimum value from each row
    and then from each column. Returns the total reduction cost.
    """
    n = len(matrix)
    reduction_cost = 0

    # 1. Row reduction
    for i in range(n):
        row_min = min(matrix[i])
        if row_min != float('inf') and row_min > 0:
            reduction_cost += row_min
            for j in range(n):
                if matrix[i][j] != float('inf'):
                    matrix[i][j] -= row_min

    # 2. Column reduction
    for j in range(n):
        col_min = min(matrix[i][j] for i in range(n))
        if col_min != float('inf') and col_min > 0:
            reduction_cost += col_min
            for i in range(n):
                if matrix[i][j] != float('inf'):
                    matrix[i][j] -= col_min

    return reduction_cost

def solve_tsp_branch_and_bound(cost_matrix):
    """
    Solves the Traveling Salesperson Problem using Branch and Bound.
    """
    n = len(cost_matrix)

    # Create the root node by reducing the initial matrix
    initial_matrix = copy.deepcopy(cost_matrix)
    initial_bound = reduce_matrix(initial_matrix)

    # Priority Queue to store the active nodes (Min-Heap based on bound)
    pq = []
    root = Node(initial_matrix, path=[0], bound=initial_bound, level=0, current_city=0)
    heapq.heappush(pq, root)

    while pq:
        # Extract the node with the lowest bound (Best-First Search)
        curr_node = heapq.heappop(pq)

        # Base Case: If we have visited all n cities
        if curr_node.level == n - 1:
            # The final edge returning to the start city is already accounted for in the bound
            curr_node.path.append(0)
            return curr_node.path, curr_node.bound

        # Try visiting all unvisited cities from the current city
        for v in range(n):
            if v not in curr_node.path:
                # 1. Create a copy of the parent's reduced matrix
                child_matrix = copy.deepcopy(curr_node.matrix)

                # 2. Get the cost of moving from current_city to v
                edge_cost = curr_node.matrix[curr_node.current_city][v]

                # 3. Prevent revisiting the current row and column by setting them to infinity
                for i in range(n):
                    child_matrix[curr_node.current_city][i] = float('inf')
                    child_matrix[i][v] = float('inf')

                # 4. Prevent premature cycles (returning to the start city too early)
                child_matrix[v][0] = float('inf')

                # 5. Reduce the new child matrix
                child_reduction_cost = reduce_matrix(child_matrix)

                # 6. Calculate the lower bound for the child node
                child_bound = curr_node.bound + edge_cost + child_reduction_cost

                # 7. Create the child node and push it to the priority queue
                child_path = curr_node.path + [v]
                child_node = Node(child_matrix, child_path, child_bound, curr_node.level + 1, v)

                heapq.heappush(pq, child_node)

    return [], -1

if __name__ == '__main__':
    INF = float('inf')

    # 5-City TSP Cost Matrix
    # Distances between cities. Diagonal is INF because a city cannot connect to itself.
    tsp_matrix = [
        [INF, 20,  30,  10,  11],
        [15,  INF, 16,  4,   2],
        [3,   5,   INF, 2,   4],
        [19,  6,   18,  INF, 3],
        [16,  4,   7,   16,  INF]
    ]

    print("--- TSP using Branch and Bound ---")
    print("5-City Cost Matrix:")
    for row in tsp_matrix:
        print(["INF" if x == INF else f"{x:2}" for x in row])

    print("\nSolving...")
    optimal_path, optimal_cost = solve_tsp_branch_and_bound(tsp_matrix)

    if optimal_cost != -1:
        path_str = " -> ".join(map(str, optimal_path))
        print(f"Optimal Hamiltonian Cycle: {path_str}")
        print(f"Minimum Total Cost:        {optimal_cost}")
    else:
        print("No valid tour exists.")

--- TSP using Branch and Bound ---
5-City Cost Matrix:
['INF', '20', '30', '10', '11']
['15', 'INF', '16', ' 4', ' 2']
[' 3', ' 5', 'INF', ' 2', ' 4']
['19', ' 6', '18', 'INF', ' 3']
['16', ' 4', ' 7', '16', 'INF']

Solving...
Optimal Hamiltonian Cycle: 0 -> 3 -> 1 -> 4 -> 2 -> 0
Minimum Total Cost:        28
